<a href="https://colab.research.google.com/github/AhalaAyyalas/MachineLearning/blob/main/FPv2_parts12345.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Installation of libraries and modules

!pip install requests
!pip install chandassu regex --break-system-packages
!pip install transformers torch -q
!pip install librosa -q
import librosa
import numpy as np
import warnings
import requests
import os
import sys
from datetime import datetime
from pathlib import Path
from transformers import pipeline
from chandassu.telugu.laghuvu_guruvu import LaghuvuGuruvu
from chandassu.telugu.padya_bhedam import check_padyam
from google.colab import userdata

# PART1 - ATT


In [ ]:
#@title API from Sarvam.ai

#@title API from Sarvam.ai

from google.colab import userdata

# ─── CONFIGURATION ────────────────────────────────────────────────────────────

API_KEY = userdata.get("SARVAM_API_KEY")

SARVAM_STT_URL = "https://api.sarvam.ai/speech-to-text"

SUPPORTED_FORMATS = {
    ".mp3": "audio/mpeg",
    ".mp4": "audio/mp4",
    ".wav": "audio/wav",
    ".m4a": "audio/mp4",
    ".ogg": "audio/ogg",
    ".flac": "audio/flac",
    ".webm": "audio/webm",
}


# ─── CORE TRANSCRIPTION FUNCTION ──────────────────────────────────────────────

def transcribe_telugu(audio_path: str, translate: bool = False, save_txt: bool = True) -> str:
    """
    Transcribe a Telugu audio file using Sarvam AI's Saaras V3 model.

    Args:
        audio_path : Path to the audio/video file
        translate  : If True, also translates output to English
        save_txt   : If True, saves transcription to a .txt file

    Returns:
        Transcribed text string
    """
    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise ValueError(f"❌ File not found: {audio_path}")

    ext = audio_path.suffix.lower()
    if ext not in SUPPORTED_FORMATS:
        raise ValueError(f"❌ Unsupported format: {ext}\n   Supported: {', '.join(SUPPORTED_FORMATS.keys())}")

    file_size_mb = audio_path.stat().st_size / (1024 * 1024)
    if file_size_mb > 25:
        raise ValueError(f"❌ File too large: {file_size_mb:.1f} MB (max 25 MB)\n   Tip: Split the file using ffmpeg:\n   ffmpeg -i input.mp4 -t 600 -c copy part1.mp4")

    api_key = API_KEY if API_KEY != "YOUR_SARVAM_API_KEY_HERE" else os.getenv("SARVAM_API_KEY")
    if not api_key:
        raise ValueError("❌ No API key found!\n   Get your FREE key at: https://dashboard.sarvam.ai\n   Then set it in Colab secrets as SARVAM_API_KEY")

    print(f"📂 File     : {audio_path.name} ({file_size_mb:.2f} MB)")
    print(f"🤖 Model    : Sarvam Saaras V3 (Telugu)")
    print(f"⏳ Transcribing...")

    mime_type = SUPPORTED_FORMATS[ext]

    with open(audio_path, "rb") as f:
        response = requests.post(
            SARVAM_STT_URL,
            headers={"api-subscription-key": api_key},
            files={"file": (audio_path.name, f, mime_type)},
            data={
                "language_code": "te-IN",
                "model": "saaras:v3",
                "with_timestamps": "true",
                "with_disfluencies": "false",
            }
        )

    if response.status_code != 200:
        raise ValueError(f"❌ API Error {response.status_code}: {response.text}")

    data = response.json()
    transcript = data.get("transcript", "").strip()

    print("\n" + "=" * 60)
    print("✅ TRANSCRIPTION (Telugu)")
    print("=" * 60)
    print(transcript)
    print("=" * 60)

    words = data.get("words", [])
    if words:
        print("\n📋 WORD-LEVEL TIMESTAMPS:")
        print("-" * 40)
        for w in words:
            word  = w.get("word", "")
            start = w.get("start", 0)
            end   = w.get("end", 0)
            print(f"  [{start:.2f}s → {end:.2f}s]  {word}")

    english_translation = None
    if translate and transcript:
        print("\n⏳ Translating to English...")
        trans_response = requests.post(
            "https://api.sarvam.ai/translate",
            headers={
                "api-subscription-key": api_key,
                "Content-Type": "application/json"
            },
            json={
                "input": transcript,
                "source_language_code": "te-IN",
                "target_language_code": "en-IN",
                "speaker_gender": "Female",
                "mode": "formal"
            }
        )
        if trans_response.status_code == 200:
            english_translation = trans_response.json().get("translated_text", "")
            print("\n🌐 ENGLISH TRANSLATION:")
            print("-" * 40)
            print(english_translation)

    if save_txt:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = audio_path.parent / f"{audio_path.stem}_transcription_{timestamp}.txt"

        with open(out_path, "w", encoding="utf-8") as f:
            f.write("Telugu Audio Transcription\n")
            f.write(f"File  : {audio_path.name}\n")
            f.write(f"Model : Sarvam Saaras V3 (te-IN)\n")
            f.write(f"Date  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 60 + "\n\n")
            f.write("TELUGU TRANSCRIPT\n")
            f.write("-" * 40 + "\n")
            f.write(transcript + "\n\n")

            if english_translation:
                f.write("ENGLISH TRANSLATION\n")
                f.write("-" * 40 + "\n")
                f.write(english_translation + "\n\n")

            if words:
                f.write("WORD-LEVEL TIMESTAMPS\n")
                f.write("-" * 40 + "\n")
                for w in words:
                    f.write(f"[{w.get('start', 0):.2f}s → {w.get('end', 0):.2f}s]  {w.get('word', '')}\n")

        print(f"\n💾 Saved to : {out_path}")

    return transcript


# ─── BATCH FUNCTION ───────────────────────────────────────────────────────────

def batch_transcribe(folder_path: str, translate: bool = False):
    """Transcribe all supported audio files in a folder."""
    folder = Path(folder_path)
    files  = [f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED_FORMATS]

    if not files:
        print(f"No supported audio files found in: {folder_path}")
        return

    print(f"Found {len(files)} audio file(s) in '{folder_path}'\n")
    all_results = {}

    for f in sorted(files):
        print(f"\n{'─' * 50}")
        print(f"Processing: {f.name}")
        text = transcribe_telugu(str(f), translate=translate, save_txt=True)
        all_results[f.name] = text

    summary_path = folder / f"batch_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("Batch Telugu Transcription — Sarvam AI\n")
        f.write(f"Date  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Files : {len(all_results)}\n")
        f.write("=" * 60 + "\n\n")
        for fname, text in all_results.items():
            f.write(f"FILE: {fname}\n")
            f.write("-" * 40 + "\n")
            f.write(text + "\n\n")

    print(f"\n✅ Batch summary saved to: {summary_path}")


In [ ]:
#@title Transcribing the Audio

transcribed = transcribe_telugu("srikaivalya.mp4", translate=False, save_txt=False)

# PART2 - U or I?

In [ ]:
#@title U,I classification

lg = LaghuvuGuruvu(data=transcribed).generate()

# Returns list of (aksharam, symbol) tuples
# symbol is "U" for Guruvu, "|" for Laghuvu
print(lg)

# Converting to U/I notation:
sequence = "".join("U" if s == "U" else "I" for _, s in lg)
print(sequence)  # IIUUI... etc.

# PART3 - Which Chandassu?

In [ ]:
#@title Chandassu (Padyam) Identification

TYPES = [
    "vutpalamaala",
    "champakamaala",
    "mattebhamu",
    "saardulamu",
    "kandamu",
    "aataveladi",
    "teytageethi",
    "seesamu",
]

scores = {}
results = {}
for t in TYPES:
    r = check_padyam(lg_data=lg_data, type=t, return_micro_score=True, verbose=False)
    scores[t] = r["chandassu_score"]
    results[t] = r

top3 = sorted(scores.items(), key=lambda x: -x[1])[:3]
print("=== Top 3 Matches ===")
for i, (t, _) in enumerate(top3, 1):
    result = results[t]  # no second call
    print(f"\n#{i} {t}  —  {result['chandassu_score']:.0%}")
    for criterion, score in result["micro_score"].items():
        bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
        print(f"  {criterion:<18s} {bar}  {score:.0%}")

# PART4 - Sentiment Analysis (text)

In [ ]:
#@title On the text

classifier = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis"
)

text = [transcribed]

for a in text:
  result = classifier(transcribed)[0]
  print(f"Text: {transcribed[:60]}...")
  print(f"Sentiment: {result['label']} | Score: {result['score']:.2f}")

#PART5 - Speech Emotion Recognition (audio)

In [ ]:
#@title On the audio

warnings.filterwarnings("ignore")

y, sr = librosa.load("srikaivalya.mp4")

pitch, _ = librosa.piptrack(y=y, sr=sr)
energy = np.mean(librosa.feature.rms(y=y))
tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
tempo = float(np.mean(tempo))  # ← add this
avg_pitch = np.mean(pitch[pitch > 0])

print(f"⚡ Energy    : {energy:.4f}  → {'high' if energy > 0.05 else 'low'}")
print(f"🎵 Avg Pitch : {avg_pitch:.2f} Hz → {'high' if avg_pitch > 200 else 'low'}")
print(f"🥁 Tempo     : {tempo:.1f} BPM → {'fast' if tempo > 120 else 'slow'}")

# Simple rule-based sentiment from audio properties
if energy > 0.05 and avg_pitch > 200:
    print("🟢 Audio Sentiment: Positive / Energetic")
elif energy < 0.02:
    print("🔵 Audio Sentiment: Calm / Neutral")
else:
    print("🔴 Audio Sentiment: Negative / Tense")